# Maximum Gram approximation error versus K

This is the retained experiment: $M=200$ independent samples, each containing $n=200$ observations from $N(0,1)$, with $\Omega=450$ and $N=50$ geometrically spaced bandwidths in $[0.01,2]$. Each sample is kept fixed across all $K$ values.

The single plotted error curve is

$$E_K^{\max}=\max_{1\le m\le200}\max_{1\le r,s\le50}\left|G_{rs}^{(m)}-G_{rs}^{(m),450,K}\right|.$$

It takes the maximum over both the 200 realizations and all bandwidth pairs, including diagonal entries. It measures the total approximation error, including truncation and quadrature. The dashed line marks the $10^{-8}$ target; the dotted line marks $K=1536$.

The analytic reference is implemented in [density.py](../kde_tools/density.py), from the thesis's Equation `eq:beta2_exact_formula02`. Approximations call the existing [frequency quadrature implementation](../kde_tools/fourier/similarities.py), with $K$ intervals and $K+1$ nodes. The seed rule is `20261024 + sample_index`, with zero-based indexing. Results are empirical over these 200 samples, rather than a uniform error guarantee.

Previously computed results are reused only after checking matching numerical settings, actual samples, bandwidths, and K values. Otherwise, the notebook runs the experiment. Every displayed figure is automatically saved as PDF under `../outputs/pdfs/g_rs_error_vs_k_repeated_samples/`. Settings, reports, maximum-error summaries, and PNG previews are saved under `../outputs/`. Matching summary views share the actual samples, bandwidths, exact Gram matrices, and per-sample errors; `cache_reference.json` locates those numerical files relative to each view's data folder.

In [ ]:
from pathlib import Path
import json
import hashlib
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from time import perf_counter
import sys

code_directory = next(
    (candidate.resolve()
     for ancestor in (Path.cwd(), *Path.cwd().parents)
     for candidate in (ancestor, ancestor / 'Code' / 'Code_kde')
     if (candidate / 'kde_tools' / 'pdf_export.py').is_file()),
    None,
)
if code_directory is None:
    raise FileNotFoundError('Run this notebook from inside the MASTERARBEIT project.')
if str(code_directory) not in sys.path:
    sys.path.insert(0, str(code_directory))

import matplotlib.pyplot as plt
import numpy as np
from threadpoolctl import threadpool_limits

from kde_tools.density import exact_gaussian_kde_l2_gram
from kde_tools.fourier import (
    FOURIER_L2_METHOD_VERSION,
    gaussian_kde_l2_gram_fourier,
)
from kde_tools.pdf_export import enable_automatic_pdf_saving

notebook_name = 'g_rs_error_vs_k_repeated_samples'
pdf_output_directory = enable_automatic_pdf_saving(notebook_name)

In [ ]:
n = 200
realizations = 200
seed = 20261024
N = 50
h_min, h_max = 0.01, 2.0
frequency_cutoff = 450.0
K_candidates = np.arange(64, 2304 + 1, 16, dtype=int)
highlight_K = 1536
absolute_tolerance = 1e-8
workers = min(4, os.cpu_count() or 1)

samples = np.stack([
    np.random.default_rng(seed + m).normal(size=n)
    for m in range(realizations)
])
bandwidths = np.geomspace(h_min, h_max, N)
settings = {
    'sample_distribution': 'standard_normal', 'realizations': realizations,
    'n': n, 'seed': seed, 'seed_rule': 'seed + zero_based_sample_index',
    'N': N, 'h_min': h_min, 'h_max': h_max, 'bandwidth_grid': 'geometric',
    'frequency_cutoff': frequency_cutoff, 'K_candidates': K_candidates.tolist(),
    'highlight_K': highlight_K, 'absolute_tolerance': absolute_tolerance,
    'K_convention': 'trapezoidal intervals; frequency_points = K + 1',
    'per_sample_error': 'max over all r,s including diagonal of abs(G_exact - G_approx)',
    'summary_curves': ['maximum of per-sample errors'],
    'reference': 'exact_gaussian_kde_l2_gram_float64',
    'method_version': FOURIER_L2_METHOD_VERSION,
    'numpy_version': np.__version__,
}
configuration_id = hashlib.sha256(json.dumps(settings, sort_keys=True).encode()).hexdigest()[:12]
data_directory = code_directory / 'outputs' / 'data' / notebook_name / configuration_id
image_directory = code_directory / 'outputs' / 'images' / notebook_name
data_directory.mkdir(parents=True, exist_ok=True)
image_directory.mkdir(parents=True, exist_ok=True)
(data_directory / 'settings.json').write_text(json.dumps(settings, indent=2) + '\n')
print(f'Configuration {configuration_id}: {realizations} independent samples of size {n}.')
print(f'Omega={frequency_cutoff:g}; {N} geometric bandwidths in [{h_min:g}, {h_max:g}].')
print(f'{len(K_candidates)} K values; up to {workers} calculation threads.')

In [ ]:
def calculate_one_sample(sample_index):
    current_sample = samples[sample_index]
    reference = exact_gaussian_kde_l2_gram(current_sample, bandwidths)
    local_errors = np.empty(len(K_candidates))
    local_worst_indices = np.empty((len(K_candidates), 2), dtype=int)
    for k_index, K in enumerate(K_candidates):
        approximate = gaussian_kde_l2_gram_fourier(
            current_sample, bandwidths,
            frequency_cutoff=frequency_cutoff,
            frequency_points=int(K) + 1,
        )
        difference = np.abs(approximate - reference)
        r, s = np.unravel_index(np.argmax(difference), difference.shape)
        local_errors[k_index] = difference[r, s]
        local_worst_indices[k_index] = (r, s)
    return sample_index, reference, local_errors, local_worst_indices


# Aggregation choices do not change the per-sample numerical calculation.
calculation_settings = {key: value for key, value in settings.items() if key != 'summary_curves'}
cached_directory = None
for candidate in sorted(data_directory.parent.iterdir()):
    settings_path = candidate / 'settings.json'
    required_paths = [candidate / name for name in (
        'per_sample_results.npz', 'exact_grams.npy', 'samples.npy', 'bandwidths.npy',
    )]
    if not settings_path.is_file() or not all(p.is_file() for p in required_paths):
        continue
    candidate_settings = json.loads(settings_path.read_text())
    candidate_calculation_settings = {
        key: value for key, value in candidate_settings.items() if key != 'summary_curves'
    }
    if candidate_calculation_settings != calculation_settings:
        continue
    if (not np.array_equal(np.load(candidate / 'samples.npy'), samples)
            or not np.array_equal(np.load(candidate / 'bandwidths.npy'), bandwidths)):
        continue
    with np.load(candidate / 'per_sample_results.npz') as cached:
        if (not np.array_equal(cached['K'], K_candidates)
                or cached['errors'].shape != (realizations, len(K_candidates))
                or cached['worst_indices'].shape != (realizations, len(K_candidates), 2)
                or not np.isfinite(cached['errors']).all()
                or np.any(cached['errors'] < 0)):
            continue
        cached_errors = cached['errors'].copy()
        cached_worst_indices = cached['worst_indices'].copy()
    cached_references = np.load(candidate / 'exact_grams.npy')
    if cached_references.shape != (realizations, N, N) or not np.isfinite(cached_references).all():
        continue
    cached_directory = candidate
    break

if cached_directory is not None:
    per_sample_errors = cached_errors
    worst_indices = cached_worst_indices
    exact_grams = cached_references
    elapsed_seconds = 0.0
    calculation_directory = cached_directory
    print(f'Reused all {realizations} verified samples from {cached_directory.name}; no simulations rerun.')
else:
    calculation_directory = data_directory
    np.save(calculation_directory / 'samples.npy', samples)
    np.save(calculation_directory / 'bandwidths.npy', bandwidths)
    per_sample_errors = np.empty((realizations, len(K_candidates)))
    worst_indices = np.empty((realizations, len(K_candidates), 2), dtype=int)
    exact_grams = np.empty((realizations, N, N))
    start = perf_counter()
    with threadpool_limits(limits=1), ThreadPoolExecutor(max_workers=workers) as executor:
        futures = [executor.submit(calculate_one_sample, m) for m in range(realizations)]
        for completed, future in enumerate(as_completed(futures), start=1):
            sample_index, reference, local_errors, local_worst = future.result()
            exact_grams[sample_index] = reference
            per_sample_errors[sample_index] = local_errors
            worst_indices[sample_index] = local_worst
            if completed % 10 == 0 or completed == realizations:
                print(f'Completed {completed}/{realizations} samples ({perf_counter() - start:.1f} s).', flush=True)
    elapsed_seconds = perf_counter() - start
    np.save(data_directory / 'exact_grams.npy', exact_grams)
    np.savez_compressed(
        data_directory / 'per_sample_results.npz',
        K=K_candidates, errors=per_sample_errors, worst_indices=worst_indices,
    )
# Summary views point to one verified numerical cache instead of copying it.
(data_directory / 'cache_reference.json').write_text(
    json.dumps({'relative_directory': os.path.relpath(calculation_directory, data_directory)}, indent=2) + '\n'
)


In [ ]:
maximum_errors = per_sample_errors.max(axis=0)
worst_sample_indices = per_sample_errors.argmax(axis=0)
exceeding_counts = (per_sample_errors > absolute_tolerance).sum(axis=0)
summary_table = np.column_stack([
    K_candidates, maximum_errors, worst_sample_indices, exceeding_counts,
])
np.savetxt(
    data_directory / 'g_rs_error_vs_k_maximum_summary.csv', summary_table,
    delimiter=',', comments='',
    header='K,max_E_K,worst_sample_zero_based,samples_exceeding_tolerance',
    fmt=['%d', '%.17g', '%d', '%d'],
)
highlight_index = int(np.flatnonzero(K_candidates == highlight_K)[0])
passing_maximum = np.flatnonzero(maximum_errors <= absolute_tolerance)
report = {
    'configuration_id': configuration_id,
    'n': n, 'realizations': realizations, 'N': N, 'frequency_cutoff': frequency_cutoff,
    'absolute_tolerance': absolute_tolerance,
    'first_tested_K_max_below_tolerance': int(K_candidates[passing_maximum[0]]) if len(passing_maximum) else None,
    'highlight_K': highlight_K,
    'max_error_at_highlight_K': float(maximum_errors[highlight_index]),
    'samples_exceeding_tolerance_at_highlight_K': int(exceeding_counts[highlight_index]),
    'largest_tested_K': int(K_candidates[-1]),
    'max_error_at_largest_K': float(maximum_errors[-1]),
    'calculation_seconds_this_run': elapsed_seconds, 'workers': workers,
    'reused_cache_configuration': cached_directory.name if cached_directory is not None else None,
    'empirical_scope': 'maximum over 200 samples of size 200 and all pairs of 50 bandwidths including diagonal',
}
(data_directory / 'report.json').write_text(json.dumps(report, indent=2) + '\n')
print(f'First tested K with maximum E_K <= {absolute_tolerance:g}: {report["first_tested_K_max_below_tolerance"]}')
print(f'At K={highlight_K}: maximum error={maximum_errors[highlight_index]:.8e}.')
print(f'Samples above the target at K={highlight_K}: {exceeding_counts[highlight_index]}/{realizations}.')

In [ ]:
with plt.rc_context({
    'font.family': 'DejaVu Sans', 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False, 'pdf.fonttype': 42,
}):
    fig, ax = plt.subplots(figsize=(8.0, 4.8), layout='constrained')
    ax.plot(K_candidates, maximum_errors, color='#2266a5', linewidth=2.0,
            label=rf'Maximum over {realizations} samples')
    ax.axhline(absolute_tolerance, color='#333333', linestyle='--', linewidth=1.3,
               label=r'Target $10^{-8}$')
    ax.axvline(highlight_K, color='#777777', linestyle=':', linewidth=1.2,
               label=rf'Current $K={highlight_K}$')
    ax.set_yscale('log')
    ax.set_xlim(0, K_candidates[-1])
    ax.set_xticks([0, 512, 1024, 1536, 2048])
    ax.set_xlabel(r'Number of trapezoidal intervals $K$')
    ax.set_ylabel('Maximum absolute Gram error')
    ax.set_title(
        rf'Maximum error over {realizations} samples at $\Omega=450$' + '\n'
        + rf'$n={n},\ N={N},\ h\in[{h_min:g},{h_max:g}]$; geometric bandwidth grid',
        fontsize=12, pad=12,
    )
    ax.grid(axis='y', which='major', color='#e5e5e5', linewidth=0.7)
    ax.set_axisbelow(True)
    ax.legend(loc='upper right', frameon=False, fontsize=9)
    png_path = image_directory / 'g_rs_error_vs_k_maximum.png'
    fig.savefig(png_path, dpi=200, bbox_inches='tight')
    plt.show()  # Also saves the PDF through kde_tools.pdf_export.